In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *

In [2]:
# Load COMPAS data
features, target, scaler, feature_columns, categorical_columns, numerical_columns, one_hot_encode_features, features_ranges, features_type = load_compas()
# features = features.to_numpy()
# target = target.to_numpy()

# Train a logistic regression model
model, X_train, X_test, y_train, y_test = train_compas_model(features, target)

# Find the first negative instance from the model predictions on the test set
negative_instances = X_test[model.predict(X_test) == 0]
x = negative_instances[0]  # First negative instance

d:\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [3]:
from ugce import *

iea = UGCE(model, scaler, feature_columns, categorical_columns, \
            numerical_columns, one_hot_encode_features, features_ranges, features_type)

In [4]:
constraints = {
  # 'sex':'-',
  'race':'-'
}

best_individuals = iea.explain_instance(x, dynamic_constraints=True, constraints=constraints,\
                    initial_population_variability=0.5, data_distribution=True,\
                    num_generations=10, population_size=100,  early_stopping_iterations=4,
                    num_parents=10, selection_method="tournament", tournsize=20,\
                    elite_ratio=0.6, cxpb=0.7, mutpb=0.6)
best_individuals

Explaining instance {'sex': 1.0, 'age': 23.0, 'juv_fel_count': 0.0, 'juv_misd_count': 0.0, 'juv_other_count': 0.0, 'priors_count': 3.0, 'c_charge_degree': 0.0, 'race_0': 1.0, 'race_1': 0.0, 'race_2': 0.0, 'race_3': 0.0, 'race_4': 0.0, 'race_5': 0.0}
Count of identical individuals: 3.0
Constraints: {}
Immutable features: [7, 8, 9, 10, 11, 12]
Initial population average fitness: 713.1396431753269, max fitness: 1154.8788421642328
Starting evolution...
Generation 0
Generation Seed number: 42
10
Crossover Seed number: 43
Crossover Seed number: 44
Crossover Seed number: 45
Crossover Seed number: 46
Crossover Seed number: 47
Crossover Seed number: 48


IndexError: list index out of range

In [5]:
model.predict(scaler.transform(np.array(best_individuals).reshape(1, -1)))

d:\PythonEnvs\DynamicCFEs\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


array([0])

# Tests

In [ ]:
inverse_transform_individual(x, scaler, feature_columns)

{'sex': 1.0,
 'age': 23.0,
 'juv_fel_count': 0.0,
 'juv_misd_count': 0.0,
 'juv_other_count': 0.0,
 'priors_count': 3.0,
 'c_charge_degree': 0.0,
 'race_0': 1.0,
 'race_1': 0.0,
 'race_2': 0.0,
 'race_3': 0.0,
 'race_4': 0.0,
 'race_5': 0.0}

In [22]:
iea.mutpb = 0.6
iea.immutables = []
iea.constraints = {}
iea.mutate_individual(x), iea.mutate_individual(x)

42
42


(array([ 1., 49.,  3.,  0.,  0., 27.,  0.,  0.,  0.,  1.,  0.,  0.,  0.]),
 array([ 1., 49.,  3.,  0.,  0., 27.,  0.,  0.,  0.,  1.,  0.,  0.,  0.]))

In [32]:
constraints = {}
def mutate_individual( individual):
    # set_seed(seed_number + gen)  # Seed for mutation
    random.seed(43)
    np.random.seed(43)
    for i in range(len(individual)):
        if random.random() > 0.1:
            feature_name = feature_columns[i]
            if i in []:
                continue
            if feature_name in categorical_columns:
                possible_values = features_ranges[feature_name]
                original_value = individual[i]
                new_value = original_value
                while new_value == original_value:
                    new_value = random.choice(possible_values)
                individual[i] = new_value
            elif feature_name in one_hot_encode_features:
                one_hot_group = [f for f in one_hot_encode_features if f.startswith(feature_name.split('_')[0])]
                current_index = next(idx for idx in one_hot_group if individual[list(feature_columns).index(idx)] == 1)
                chosen_feature = current_index
                while chosen_feature == current_index:
                    chosen_feature = random.choice(one_hot_group)
                for one_hot_feature in one_hot_group:
                    index = list(feature_columns).index(one_hot_feature)
                    individual[index] = 1 if one_hot_feature == chosen_feature else 0
            else:
                original_value = individual[i]
                new_value = original_value
                if constraints.get(i):
                    lower, upper = constraints[i]
                    if True:
                        lower_data_distribution, upper_data_distribution = features_ranges[feature_name]
                        if lower < lower_data_distribution or upper > upper_data_distribution:
                            print(f"Constraints for {feature_name} violate the data distribution: [{lower}, {upper}] vs [{lower_data_distribution}, {upper_data_distribution}]")
                            sys.exit()
                else:
                    lower, upper = features_ranges[feature_name]
                if features_type[feature_name] == 'int':
                    while new_value == original_value:
                        new_value = random.randint(lower, upper)
                else:
                    while new_value == original_value:
                        new_value = random.uniform(lower, upper)
                individual[i] = new_value
    return individual

mutate_individual(x.copy()), mutate_individual(x.copy())

(array([ 1., 36., 11., 12.,  0., 38.,  0.,  0.,  0.,  0.,  1.,  0.,  0.]),
 array([ 1., 36., 11., 12.,  0., 38.,  0.,  0.,  0.,  0.,  1.,  0.,  0.]))

In [24]:
inverse_transform_individual(x, scaler, feature_columns)

{'sex': 1.0,
 'age': 3840.0000000000005,
 'juv_fel_count': 60.0,
 'juv_misd_count': 0.0,
 'juv_other_count': 0.0,
 'priors_count': 1026.0,
 'c_charge_degree': 0.0,
 'race_0': 0.0,
 'race_1': 0.0,
 'race_2': 1.0,
 'race_3': 0.0,
 'race_4': 0.0,
 'race_5': 0.0}

In [8]:
iea.setup_constraints()
print(iea.immutables)

[0, 7, 8, 9, 10, 11, 12]
